# Brain Tumour MRI Classifier: 4 classes (PyTorch)
Classes: **glioma, meningioma, pituitary, no tumour**. After training, the same predictions are also collapsed into a *tumour vs no tumour* detector score.

Data flow: `Training/` is split into train + validation (used for tuning and early stopping); `Testing/` is touched **once**, at the very end.

In [ ]:
!pip install -q kagglehub

In [ ]:
import random
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

# ---- Config ----
SEED       = 42
BATCH_SIZE = 32
MODEL      = "resnet18"   # "cnn" = small net from scratch | "resnet18" = pretrained, fine-tuned
# per-model settings: image size, learning rate, max epochs (early stopping usually ends sooner)
IMG_SIZE, LR, EPOCHS = {"cnn": (128, 1e-3, 60), "resnet18": (224, 3e-4, 25)}[MODEL]
PATIENCE   = 6       # epochs without val-loss improvement before stopping
VAL_FRAC   = 0.15
CKPT       = f"best_{MODEL}_4class.pt"

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Data

In [ ]:
root = Path(kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset"))
train_dir, test_dir = root / "Training", root / "Testing"
assert train_dir.is_dir() and test_dir.is_dir(), f"Unexpected dataset layout in {root}"

norm = transforms.Normalize([0.5], [0.5])
base = [transforms.Grayscale(1), transforms.Resize((IMG_SIZE, IMG_SIZE))]
train_tf = transforms.Compose(base + [transforms.RandomHorizontalFlip(),
                                      transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
                                      transforms.ColorJitter(brightness=0.2, contrast=0.2),
                                      transforms.ToTensor(), norm])
eval_tf  = transforms.Compose(base + [transforms.ToTensor(), norm])

# Two views of the same training folder: augmented for training, clean for validation
train_full = datasets.ImageFolder(train_dir, train_tf)
val_full   = datasets.ImageFolder(train_dir, eval_tf)
test_ds    = datasets.ImageFolder(test_dir,  eval_tf)

class_names = train_full.classes
NUM_CLASSES = len(class_names)
assert class_names == test_ds.classes, "Training/Testing class folders differ"
print("Classes:", train_full.class_to_idx)

# Stratified split keeps every class represented in both train and validation
train_idx, val_idx = train_test_split(
    np.arange(len(train_full)), test_size=VAL_FRAC,
    stratify=train_full.targets, random_state=SEED)

train_ds = Subset(train_full, train_idx)
val_ds   = Subset(val_full,   val_idx)

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=device.type == "cuda")
train_loader = DataLoader(train_ds, shuffle=True,  generator=torch.Generator().manual_seed(SEED), **loader_kw)
val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kw)
test_loader  = DataLoader(test_ds,  shuffle=False, **loader_kw)

print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")
print("Train class counts:", dict(zip(class_names, np.bincount(np.array(train_full.targets)[train_idx]))))

In [ ]:
# Quick look at one training batch
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, img, lab in zip(axes, images, labels):
    ax.imshow((img[0] * 0.5 + 0.5).numpy(), cmap="gray")   # undo normalisation
    ax.set_title(class_names[lab]); ax.axis("off")
plt.show()

## 2. Model
Set `MODEL` in the config cell:
* `"cnn"`: small net from scratch, 4 × (Conv → BatchNorm → ReLU → MaxPool), global average pooling, one **logit per class**
* `"resnet18"`: ImageNet-pretrained ResNet18; the first conv is collapsed to 1 channel (weights summed over RGB) so the grayscale pipeline stays unchanged

In [ ]:
def block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
        nn.BatchNorm2d(c_out), nn.ReLU(inplace=True), nn.MaxPool2d(2))

class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(block(1, 16), block(16, 32), block(32, 64), block(64, 128),
                                      nn.AdaptiveAvgPool2d(1))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))   # raw logits, shape (B, num_classes)

def build_model(name, num_classes):
    if name == "cnn":
        return CNN(num_classes)
    if name == "resnet18":
        net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        w = net.conv1.weight.data.sum(dim=1, keepdim=True)          # RGB -> grayscale
        net.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        net.conv1.weight.data = w
        net.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(net.fc.in_features, num_classes))
        return net
    raise ValueError(f"Unknown MODEL: {name}")

model = build_model(MODEL, NUM_CLASSES).to(device)
print(f"{MODEL}: {sum(p.numel() for p in model.parameters()):,} parameters")

## 3. Training
* `CrossEntropyLoss` (softmax is built in, so the model outputs raw logits)
* LR scheduler + early stopping on validation loss; the best weights are saved and reloaded

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)

def run_epoch(loader, train):
    model.train(train)
    total_loss = correct = n = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y)
            correct    += (logits.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val, stale = float("inf"), 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)
    scheduler.step(va_loss)
    for k, v in zip(history, (tr_loss, va_loss, tr_acc, va_acc)):
        history[k].append(v)

    improved = va_loss < best_val
    if improved:
        best_val, stale = va_loss, 0
        torch.save(model.state_dict(), CKPT)
    else:
        stale += 1

    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.3f}{'  *' if improved else ''}")
    if stale >= PATIENCE:
        print(f"Early stopping (no val-loss improvement for {PATIENCE} epochs)")
        break

In [ ]:
epochs  = np.arange(1, len(history["val_loss"]) + 1)
best_ep = int(np.argmin(history["val_loss"])) + 1        # saved-checkpoint epoch (1-based)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, key, title, ylabel in ((axes[0], "loss", "Loss vs. Epoch", "Loss"),
                               (axes[1], "acc",  "Accuracy vs. Epoch", "Accuracy")):
    ax.plot(epochs, history[f"train_{key}"], marker="o", ms=3, label="train")
    ax.plot(epochs, history[f"val_{key}"],   marker="o", ms=3, label="validation")
    ax.axvline(best_ep, color="grey", linestyle="--", label=f"saved checkpoint (epoch {best_ep})")
    ax.set(title=title, xlabel="Epoch", ylabel=ylabel)
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

### Plateau check
Reads the training history and says whether accuracy has stalled, and if so what to change. It uses lowest validation *loss* to pick the checkpoint, the same rule early stopping and the saved weights use.

In [ ]:
n        = len(history["val_loss"])
best_ep  = int(np.argmin(history["val_loss"]))              # same criterion as the saved checkpoint
best_acc = history["val_acc"][best_ep]
tr_acc   = history["train_acc"][best_ep]
gap      = tr_acc - best_acc                                # negative = val ahead of train

print(f"Checkpoint: epoch {best_ep + 1} of {n} (lowest val loss)")
print(f"At that epoch: val acc {best_acc:.3f} | train acc {tr_acc:.3f} | gap {gap:.3f}")

still_improving = (n - best_ep) <= 3     # best result is very recent
if still_improving and tr_acc < 0.95:
    print("-> Still improving AND train accuracy is low (underfitting). Train longer and/or use MODEL = 'resnet18'.")
elif still_improving:
    print("-> Still improving. Raise EPOCHS / PATIENCE and continue.")
elif best_acc >= 0.97:
    print("-> Converged. Validation is near-saturated, so it can no longer rank further changes; judge by the test report below and avoid tuning against it.")
elif best_acc < 0.90:
    print("-> Plateau with LOW accuracy (underfitting). Use MODEL = 'resnet18' and IMG_SIZE = 224.")
elif gap > 0.05:
    print("-> Plateau with a large train-val gap (overfitting). Add stronger augmentation / dropout.")
else:
    print("-> Plateau at a good level with a small gap. Remaining errors are likely hard glioma/meningioma cases; check the confusion matrix below.")

## 4. Final evaluation on the held-out `Testing/` split
Validation comes from the same folder as training, so it can contain near-duplicate slices from the same patients and tends to read a few points higher. **The test numbers below are the honest ones.**

In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=device))
model.eval()

preds, y_true = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds.append(model(x.to(device)).argmax(1).cpu()); y_true.append(y)
y_pred, y_true = torch.cat(preds).numpy(), torch.cat(y_true).numpy()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=class_names,
                                        cmap="Blues", xticks_rotation=45, ax=ax)
plt.tight_layout(); plt.show()

### Tumour vs no-tumour view
The same test predictions, collapsed into the original detector question: is *any* tumour present? Confusing glioma with meningioma counts as correct here, since a tumour was still detected.

In [ ]:
healthy = train_full.class_to_idx["notumor"]
det_true, det_pred = (y_true != healthy).astype(int), (y_pred != healthy).astype(int)
print(classification_report(det_true, det_pred, target_names=["No tumour", "Tumour"], digits=4))
missed = int(((det_true == 1) & (det_pred == 0)).sum())
print(f"Tumours wrongly called 'no tumour': {missed} of {int(det_true.sum())}  <- the costly error for a detector")